# Chapter 3: whose number wins

Section 3.7 loads Kestrel's synthetic week with all three gravity sources, stores the precedence table,
picks rank 1, flags any meter gap outside the 0.0 to 0.2 band and reruns efficiency under each source.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import kestrel as k

con = k.connect()
sources = k.gravity_sources(con)
print(sources.to_string(index=False))

The signed precedence rows for wort gravity. Rank 1 is the source of record.

In [ ]:
precedence = k.q(con, "SELECT * FROM precedence WHERE measurement_code = 'WORT_GRAVITY'")
print(precedence.to_string(index=False))
record = k.gravity_of_record(con, sources)
print(record.to_string(index=False))

The gap band. Kestrel's meter normally reads 0.1 above the lab, so a gap from 0.0 to 0.2 is normal.

In [ ]:
sources["alarm"] = ~sources.gap.between(0.0 - 1e-9, 0.2 + 1e-9)
print(sources[["brew_id", "lab", "inline_meter", "gap", "alarm"]].to_string(index=False))

Section 3.3 worked through brew B26-0068 at 12.0 and 12.3 degrees Plato.

In [ ]:
theoretical = k.theoretical_extract(con)
for p in (12.0, 12.3):
    sg = k.sg_from_plato(p)
    mass = 30000 * sg * k.WATER_KG_PER_L
    ext = k.extract_kg(300, p)
    print(f"{p} degrees Plato: SG {sg:.5f}, wort mass {mass:,.0f} kg, extract {ext:,.0f} kg, "
          f"efficiency {100 * ext / theoretical['B26-0068']:.1f} percent")

Efficiency for the week under each source. The week is a sum over a sum: all the extract over all the grist.

In [ ]:
def weekly_efficiency(source):
    ext = sum(k.extract_kg(300, row[source]) for _, row in sources.iterrows())
    return 100 * ext / theoretical[sources.brew_id].sum()

for source in ("lab", "inline_meter", "brew_sheet"):
    print(f"{source:13s} {weekly_efficiency(source):.2f} percent")

Change the signed rule: put the meter at rank 1 and the efficiency every report shows moves with it.

In [ ]:
rule = {"lab": 2, "inline_meter": 1, "brew_sheet": 3}
first = min(rule, key=rule.get)
print(f"rank 1 is now {first}: weekly efficiency {weekly_efficiency(first):.2f} percent")